# Appendix P — Table 7: non-pushforward stochastic policies

This notebook reproduces the non-pushforward stochastic-intervention experiment in Appendix P. The policy changes the conditional law of D given Z, so the data-score pushforward construction is unavailable for the APE. The notebook compares Time-SMR without joint training and Time-SMR with joint training.

In [ ]:

from pathlib import Path
import sys
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO = Path.cwd()
for parent in [REPO, *REPO.parents]:
    if (parent / "src" / "genriesz" / "scorematchingriesz.py").exists():
        REPO = parent
        break
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import genriesz.scorematchingriesz as smr
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RANDOM_SEED = 123
np.random.seed(RANDOM_SEED)
print("device:", DEVICE)


In [ ]:
N_TRIALS = 200
N = 1000
N_FOLDS = 2
DELTA = 1.0
N_MC_TRUTH = 200000
HIDDEN_DIMS = (256, 256, 256)
OUTCOME_EPOCHS = 200
RATIO_STEPS = 4000
BATCH_SIZE = 256
INTEGRATION_STEPS = 200
AME_LOCAL_SHIFT = 0.05
CLIP_LOG_RATIO = 20.0
TABLE_TITLE = "Appendix P Table 7: non-pushforward stochastic policies"
FIGURE_TITLE = "Appendix P Table 7 errors"

In [ ]:

SIGMA_BASE = np.array([[1.0, 0.1, 0.1], [0.1, 1.0, 0.1], [0.1, 0.1, 1.0]])
SIGMA_ZZ = SIGMA_BASE[1:, 1:]
SIGMA_DZ = SIGMA_BASE[0:1, 1:]
COND_COEF = SIGMA_DZ @ np.linalg.inv(SIGMA_ZZ)
COND_VAR = SIGMA_BASE[0, 0] - (SIGMA_DZ @ np.linalg.inv(SIGMA_ZZ) @ SIGMA_DZ.T)[0, 0]

def sigmoid(u):
    return 1.0 / (1.0 + np.exp(-u))

def mu_function(x):
    return 1.0 + x[:,0] + 0.1*x[:,0]**2 + 2*np.sin(x[:,0]) + x[:,1] + x[:,0]*x[:,1] + x[:,2]**2 + x[:,2]**3

def partial_d_mu(x):
    return 1.0 + 0.2*x[:,0] + 2*np.cos(x[:,0]) + x[:,1]

def sample_observed(n, seed):
    rng=np.random.default_rng(seed)
    x=rng.multivariate_normal(np.zeros(3), SIGMA_BASE, size=n).astype("float32")
    y=mu_function(x)+rng.normal(size=n)
    return x.astype("float32"), y.astype("float32")

def sample_policy(n, delta, sign, seed):
    rng=np.random.default_rng(seed)
    z=rng.multivariate_normal(np.zeros(2), SIGMA_ZZ, size=n)
    m0=(z @ COND_COEF.T).reshape(-1)
    bfun=np.tanh(z[:,0]) + 0.5*np.sin(z[:,1])
    mean=m0 + sign*float(delta)*bfun
    sd=math.sqrt(COND_VAR)*(0.8 + 0.4*sigmoid(sign*z[:,0]))
    d=rng.normal(mean, sd)
    return np.column_stack([d,z]).astype("float32")

def true_ame(seed=0):
    x,_=sample_observed(N_MC_TRUTH, seed)
    return float(np.mean(partial_d_mu(x)))

def true_ape(delta, seed=0):
    xp=sample_policy(N_MC_TRUTH, delta, +1, seed+11)
    xm=sample_policy(N_MC_TRUTH, delta, -1, seed+22)
    return float(mu_function(xp).mean() - mu_function(xm).mean())


def summarize_trials(df, group_cols):
    return df.groupby(group_cols).agg(trials=("estimate", "count"), truth=("truth", "mean"), bias=("error", "mean"), mse=("error", lambda s: float(np.mean(np.square(s)))), coverage=("covered", "mean"), avg_se=("se", "mean")).reset_index()


In [ ]:

def time_ratio(method, x_q, x_p, x_eval, seed):
    if method == "Time-SMR":
        model=smr.fit_time_smr_dre_infinity(x_q, x_p, hidden_dims=HIDDEN_DIMS, n_steps=RATIO_STEPS, batch_size=BATCH_SIZE, seed=seed, device=DEVICE)
        log_r=smr.log_ratio_from_time_score(model, x_eval, steps=INTEGRATION_STEPS, normalize=True, x_p_for_norm=x_p, device=DEVICE)
    else:
        model=smr.fit_joint_smr_dre_infinity(x_q, x_p, hidden_dims=HIDDEN_DIMS, n_steps=RATIO_STEPS, batch_size=BATCH_SIZE, seed=seed, device=DEVICE)
        log_r=smr.log_ratio_from_joint_time_head(model, x_eval, steps=INTEGRATION_STEPS, normalize=True, x_p_for_norm=x_p, device=DEVICE)
    return np.exp(np.clip(log_r.detach().cpu().numpy().reshape(-1), -CLIP_LOG_RATIO, CLIP_LOG_RATIO))

def estimate_nonpush_trial(seed, theta_ame, theta_ape):
    x,y=sample_observed(N, seed)
    xplus=sample_policy(N, DELTA, +1, seed+111)
    xminus=sample_policy(N, DELTA, -1, seed+222)
    rows=[]
    for method in ["Time-SMR", "Joint-SMR"]:
        scores_ame=np.zeros(N); scores_ape=np.zeros(N)
        for train_idx, test_idx in smr.crossfit_splits(N, n_folds=N_FOLDS, seed=seed):
            x_train,y_train=x[train_idx], y[train_idx]
            x_test,y_test=x[test_idx], y[test_idx]
            outcome=smr.fit_outcome_net(x_train, y_train, hidden_dims=HIDDEN_DIMS, n_epochs=OUTCOME_EPOCHS, batch_size=BATCH_SIZE, seed=seed, device=DEVICE)
            gamma=smr.predict_outcome(outcome, x_test, device=DEVICE).reshape(-1)
            residual=y_test-gamma
            m_ame=smr.partial_d_outcome(outcome, x_test, coordinate=0, device=DEVICE).reshape(-1)
            alpha_ame=(time_ratio(method, shift_x(x_train, AME_LOCAL_SHIFT), x_train, x_test, seed) - time_ratio(method, shift_x(x_train, -AME_LOCAL_SHIFT), x_train, x_test, seed+17))/(2*AME_LOCAL_SHIFT)
            rplus=time_ratio(method, xplus[train_idx], x_train, x_test, seed+31)
            rminus=time_ratio(method, xminus[train_idx], x_train, x_test, seed+47)
            m_ape=smr.predict_outcome(outcome, xplus[test_idx], device=DEVICE).reshape(-1)-smr.predict_outcome(outcome, xminus[test_idx], device=DEVICE).reshape(-1)
            scores_ame[test_idx]=m_ame+alpha_ame*residual
            scores_ape[test_idx]=m_ape+(rplus-rminus)*residual
        for target, scores, truth in [("AME", scores_ame, theta_ame), ("APE", scores_ape, theta_ape)]:
            est=smr.wald_interval(scores)
            rows.append({"target":target, "method":method, "estimate":est.estimate, "se":est.se, "ci_low":est.ci_low, "ci_high":est.ci_high, "truth":truth, "error":est.estimate-truth, "covered":est.ci_low <= truth <= est.ci_high})
    return rows

# shifted helper reused above
def shift_x(x, delta):
    out=np.asarray(x,dtype="float32").copy(); out[:,0]+=float(delta); return out

theta_ame=true_ame(RANDOM_SEED+100)
theta_ape=true_ape(DELTA, RANDOM_SEED+101)
rows=[]
for trial in range(N_TRIALS):
    rows.extend([{**r, "trial": trial} for r in estimate_nonpush_trial(RANDOM_SEED+trial, theta_ame, theta_ape)])
nonpush_results=pd.DataFrame(rows)
print(TABLE_TITLE)
display(summarize_trials(nonpush_results, ["target", "method"]))


In [ ]:

plot_df=nonpush_results.copy()
fig, axes = plt.subplots(1, 2, figsize=(11,4), sharey=True)
for ax, target in zip(axes, ["AME", "APE"]):
    sub=plot_df[plot_df["target"]==target]
    methods=list(sub["method"].unique())
    ax.boxplot([sub.loc[sub["method"]==m,"error"] for m in methods], labels=methods, showfliers=False)
    ax.axhline(0.0, linestyle="--")
    ax.set_title(f"{FIGURE_TITLE}: {target}")
axes[0].set_ylabel("estimate minus truth")
fig.tight_layout()
plt.show()
